# PORTWISE — Final Authentication Platform
### Agentic AI for Maritime Freight Pricing and Route Optimization · Infosys Milestone 1

**One notebook only.** Run **Runtime → Run all**. No separate `app.py` upload is necessary.

This build preserves the two approved maritime backgrounds and the smooth motion from the working PORTWISE prototype. It adds signup email verification, secure password-change settings, JWT session invalidation, admin sign-in throttling, and a more informative dashboard.

**Private credentials are not included.** Configure them under Colab's **Secrets** panel (key icon). For the complete final submission, screenshots must be captured from your own live running application.

## 1 · Install dependencies

In [ ]:
!pip -q install "streamlit>=1.39,<2" "bcrypt>=4.1,<6" "PyJWT>=2.8,<3" "pandas>=2,<3" "pyngrok>=7,<8"

## 2 · Load your private Colab Secrets

Enable notebook access for `NGROK_AUTHTOKEN`, `EMAIL_ADDRESS`, `EMAIL_PASSWORD`, `ADMIN_PASSWORD` and optionally `JWT_SECRET`, `ADMIN_USERNAME`.

`EMAIL_ADDRESS` is the *single Gmail sender* for any registered Gmail/Outlook/Yahoo recipient. `EMAIL_PASSWORD` is that sender's Gmail **App Password**, not their standard password.

`REQUIRE_EMAIL_VERIFICATION=1` is the secure default. This means new accounts must verify their email before first login. You can use `0` **only for a short milestone demonstration** without an SMTP sender; do not use it for public deployment.

In [ ]:
import os, secrets
from pathlib import Path
from getpass import getpass
from google.colab import userdata

Path('/content/PORTWISE_Milestone1').mkdir(parents=True, exist_ok=True)
os.chdir('/content/PORTWISE_Milestone1')

for name in ('JWT_SECRET','NGROK_AUTHTOKEN','EMAIL_ADDRESS','EMAIL_PASSWORD','ADMIN_USERNAME','ADMIN_PASSWORD','REQUIRE_EMAIL_VERIFICATION'):
    try:
        v=userdata.get(name)
        if v:
            os.environ[name]=str(v).strip()
    except Exception:
        pass
if not os.environ.get('JWT_SECRET'):
    os.environ['JWT_SECRET']=secrets.token_urlsafe(48)
    print('Generated a private JWT secret for this runtime. Add JWT_SECRET to Colab Secrets to keep it stable across runtimes.')
if len(os.environ['JWT_SECRET']) < 32:
    raise ValueError('JWT_SECRET must be at least 32 characters long')
os.environ.setdefault('ADMIN_USERNAME','admin')
os.environ.setdefault('REQUIRE_EMAIL_VERIFICATION','1')
if not os.environ.get('ADMIN_PASSWORD'):
    print('Admin password not found. Enter one privately to enable admin login, or press Enter to skip.')
    private_value=getpass('Admin password (hidden): ')
    if len(private_value)>=12:
        os.environ['ADMIN_PASSWORD']=private_value
    elif private_value:
        print('Admin password must have at least 12 characters. Admin access remains disabled.')
if os.environ.get('EMAIL_PASSWORD'):
    os.environ['EMAIL_PASSWORD']=os.environ['EMAIL_PASSWORD'].replace(' ','')
os.environ.setdefault('DATABASE_PATH','/content/portwise_auth.db')
os.environ['SHOW_BUILD_ID']='1'
print('JWT signing:', 'ready')
print('Gmail sender:', 'configured' if os.getenv('EMAIL_ADDRESS') and os.getenv('EMAIL_PASSWORD') else 'missing')
print('Administrator:', 'configured' if os.getenv('ADMIN_PASSWORD') else 'missing')
print('Email verification on signup:', os.environ['REQUIRE_EMAIL_VERIFICATION'])
print('Ngrok:', 'configured' if os.getenv('NGROK_AUTHTOKEN') else 'token needed')

## 3 · Write the PORTWISE backend and app

In [ ]:
%%writefile auth_core.py
"""PORTWISE account store, JWT sessions and email verification.

SQLite is used for the single-instance milestone build. Never store plaintext
passwords, OTPs, recovery answers or secret values in the database.
"""
import datetime as dt
import hashlib
import hmac
import re
import secrets
import sqlite3
import time
from contextlib import contextmanager
from pathlib import Path

import bcrypt
import jwt

EMAIL_RE = re.compile(r"^[A-Za-z]{2,}[A-Za-z0-9._%+\-]*@[A-Za-z]{2,}[A-Za-z0-9\-]*(?:\.[A-Za-z]{2,}[A-Za-z0-9\-]*)+$")
QUESTIONS = [
    "What was the name of your first school?",
    "In which city were you born?",
    "What was the name of your first pet?",
    "What is the middle name of your oldest sibling?",
]
GENERIC_LOGIN_ERROR = "Invalid username/email or password."
UTC = dt.timezone.utc


def utcnow():
    return dt.datetime.now(UTC).isoformat()


def valid_email(value):
    return isinstance(value, str) and bool(EMAIL_RE.fullmatch(value.strip()))


def password_errors(password):
    issues = []
    if len(password) < 8: issues.append("at least 8 characters")
    if not re.search(r"[A-Z]", password): issues.append("one uppercase letter")
    if not re.search(r"[a-z]", password): issues.append("one lowercase letter")
    if not re.search(r"\d", password): issues.append("one number")
    if not re.search(r"[^A-Za-z0-9]", password): issues.append("one special character")
    return issues


def hashed(text):
    return bcrypt.hashpw(text.encode("utf-8"), bcrypt.gensalt()).decode("ascii")


def matches(text, saved):
    try:
        return bcrypt.checkpw(text.encode("utf-8"), saved.encode("ascii"))
    except (ValueError, TypeError, AttributeError):
        return False


def otp_digest(code, salt, secret, purpose="reset"):
    return hmac.new(secret.encode(), f"{purpose}:{salt}:{code}".encode(), hashlib.sha256).hexdigest()


class Store:
    def __init__(self, path="auth.db"):
        self.path = str(path)
        Path(self.path).parent.mkdir(parents=True, exist_ok=True)
        self.init()

    @contextmanager
    def db(self):
        conn = sqlite3.connect(self.path, timeout=20)
        conn.row_factory = sqlite3.Row
        try:
            conn.execute("PRAGMA foreign_keys=ON")
            conn.execute("PRAGMA busy_timeout=20000")
            yield conn
            conn.commit()
        except Exception:
            conn.rollback()
            raise
        finally:
            conn.close()

    def init(self):
        with self.db() as c:
            c.execute("""CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT NOT NULL UNIQUE COLLATE NOCASE,
                email TEXT NOT NULL UNIQUE COLLATE NOCASE,
                password_hash TEXT NOT NULL,
                security_question TEXT NOT NULL,
                answer_hash TEXT NOT NULL,
                created_at TEXT NOT NULL, updated_at TEXT NOT NULL,
                login_count INTEGER DEFAULT 0, last_login TEXT,
                previous_login TEXT, session_version INTEGER DEFAULT 0,
                email_verified INTEGER DEFAULT 1
            )""")
            # Upgrade existing milestone databases without dropping accounts.
            cols = {r["name"] for r in c.execute("PRAGMA table_info(users)")}
            for name, ddl in (
                ("previous_login", "TEXT"),
                ("session_version", "INTEGER NOT NULL DEFAULT 0"),
                ("email_verified", "INTEGER NOT NULL DEFAULT 1"),
            ):
                if name not in cols:
                    c.execute(f"ALTER TABLE users ADD COLUMN {name} {ddl}")
            c.execute("""CREATE TABLE IF NOT EXISTS resets (
                email TEXT PRIMARY KEY COLLATE NOCASE, code_hash TEXT NOT NULL,
                salt TEXT NOT NULL, expires_at INTEGER NOT NULL,
                attempts INTEGER NOT NULL DEFAULT 0, sent_at INTEGER NOT NULL,
                FOREIGN KEY(email) REFERENCES users(email) ON DELETE CASCADE
            )""")
            c.execute("""CREATE TABLE IF NOT EXISTS email_verifications (
                email TEXT PRIMARY KEY COLLATE NOCASE, code_hash TEXT NOT NULL,
                salt TEXT NOT NULL, expires_at INTEGER NOT NULL,
                attempts INTEGER NOT NULL DEFAULT 0, sent_at INTEGER NOT NULL,
                FOREIGN KEY(email) REFERENCES users(email) ON DELETE CASCADE
            )""")
            c.execute("""CREATE TABLE IF NOT EXISTS code_dispatch (
                email TEXT NOT NULL COLLATE NOCASE, purpose TEXT NOT NULL,
                sent_at INTEGER NOT NULL, PRIMARY KEY(email,purpose)
            )""")
            c.execute("""CREATE TABLE IF NOT EXISTS limits (
                subject TEXT PRIMARY KEY, attempts INTEGER NOT NULL,
                window_start INTEGER NOT NULL, locked_until INTEGER NOT NULL DEFAULT 0
            )""")
            c.execute("""CREATE TABLE IF NOT EXISTS audit (
                id INTEGER PRIMARY KEY AUTOINCREMENT, actor TEXT NOT NULL,
                event TEXT NOT NULL, occurred_at TEXT NOT NULL
            )""")
            c.execute("CREATE INDEX IF NOT EXISTS audit_actor_id ON audit(actor,id DESC)")

    def audit(self, actor, event):
        with self.db() as c:
            c.execute("INSERT INTO audit(actor,event,occurred_at) VALUES(?,?,?)", (str(actor)[:100],event,utcnow()))

    def limited(self, subject):
        with self.db() as c:
            r=c.execute("SELECT locked_until FROM limits WHERE subject=?", (subject,)).fetchone()
            return bool(r and r["locked_until"] > int(time.time()))

    def failure(self, subject, maximum=5, period=900):
        now=int(time.time())
        with self.db() as c:
            row=c.execute("SELECT * FROM limits WHERE subject=?",(subject,)).fetchone()
            if not row or now-row["window_start"] >= period:
                c.execute("INSERT OR REPLACE INTO limits VALUES(?,?,?,?)",(subject,1,now,0))
            else:
                n=row["attempts"]+1
                c.execute("UPDATE limits SET attempts=?, locked_until=? WHERE subject=?",
                          (n,now+period if n>=maximum else 0,subject))

    def clear_limit(self,subject):
        with self.db() as c:
            c.execute("DELETE FROM limits WHERE subject=?",(subject,))

    def signup(self, username, email, password, question, answer, *, require_verification=False):
        username=username.strip(); email=email.strip().lower()
        if not re.fullmatch(r"[\w.\-]{3,50}", username, re.UNICODE):
            return False,"Username must be 3–50 letters/digits/dots/dashes/underscores."
        if not valid_email(email): return False,"Enter a valid email address."
        if password_errors(password): return False,"Password does not meet the requirements."
        if question not in QUESTIONS or not answer.strip():
            return False,"Choose a security question and provide an answer."
        now=utcnow()
        try:
            with self.db() as c:
                c.execute("""INSERT INTO users(username,email,password_hash,security_question,
                    answer_hash,created_at,updated_at,email_verified)
                    VALUES(?,?,?,?,?,?,?,?)""",
                    (username,email,hashed(password),question,hashed(answer.strip().casefold()),now,now,
                     0 if require_verification else 1))
        except sqlite3.IntegrityError:
            with self.db() as c:
                name_taken=c.execute("SELECT 1 FROM users WHERE username=?",(username,)).fetchone()
            return False,"Username already exists." if name_taken else "Email address already registered."
        self.audit(username,"signup")
        return True,("Account created. Verify your email to activate it." if require_verification
                     else "Account created. Sign in to continue.")

    def user(self, identity):
        identity=str(identity).strip()
        with self.db() as c:
            return c.execute("SELECT * FROM users WHERE username=? OR email=?",(identity,identity)).fetchone()

    def authenticate(self, identity, password):
        key="login:"+identity.strip().casefold()
        if self.limited(key): return None,"Too many attempts. Try again in 15 minutes."
        row=self.user(identity)
        if not row or not matches(password,row["password_hash"]):
            self.failure(key)
            self.audit(identity[:100],"login_failed")
            return None,GENERIC_LOGIN_ERROR
        if not row["email_verified"]:
            return None,"Please verify your registered email before signing in."
        self.clear_limit(key)
        now=utcnow()
        with self.db() as c:
            c.execute("""UPDATE users SET previous_login=last_login,last_login=?,login_count=login_count+1
                         WHERE id=?""",(now,row["id"]))
        self.audit(row["username"],"login_success")
        return self.user(identity),None

    def change_password(self,identity,current,new):
        row=self.user(identity)
        if not row or not matches(current,row["password_hash"]):
            return False,"Current password is incorrect."
        return self.reset_password(row["email"],new)

    def reset_password(self,email,new_password):
        if password_errors(new_password): return False,"Password does not meet requirements."
        row=self.user(email)
        if not row: return False,"Account not found."
        if matches(new_password,row["password_hash"]):
            return False,"Choose a password you have not used previously."
        with self.db() as c:
            c.execute("""UPDATE users SET password_hash=?,updated_at=?,session_version=session_version+1
                         WHERE id=?""",(hashed(new_password),utcnow(),row["id"]))
            c.execute("DELETE FROM resets WHERE email=?",(row["email"],))
        self.audit(row["username"],"password_changed")
        return True,"Password updated. Sign in again."

    def security_question(self,username):
        row=self.user(username)
        return row["security_question"] if row and row["username"].casefold()==username.strip().casefold() else None

    def verify_answer(self,username,answer):
        key="question:"+username.strip().casefold()
        if self.limited(key): return False
        row=self.user(username)
        ok=bool(row and row["username"].casefold()==username.strip().casefold()
                and matches(answer.strip().casefold(),row["answer_hash"]))
        if ok: self.clear_limit(key)
        else: self.failure(key)
        return ok

    def _begin_code(self,email,secret,table,purpose):
        row=self.user(email)
        if not row or (purpose=="verify" and row["email_verified"]):
            return None,"If this address is eligible, a verification email will be sent."
        now=int(time.time())
        with self.db() as c:
            old=c.execute("SELECT sent_at FROM code_dispatch WHERE email=? AND purpose=?",
                          (row["email"],purpose)).fetchone()
            if old and now-old["sent_at"]<300:
                return None,"Please wait five minutes before requesting another code."
        code=f"{secrets.randbelow(1000000):06d}"
        salt=secrets.token_hex(16)
        with self.db() as c:
            c.execute(f"""INSERT OR REPLACE INTO {table}
                       (email,code_hash,salt,expires_at,attempts,sent_at)
                       VALUES(?,?,?,?,0,?)""",
                      (row["email"],otp_digest(code,salt,secret,purpose),salt,now+300,now))
            c.execute("INSERT OR REPLACE INTO code_dispatch(email,purpose,sent_at) VALUES(?,?,?)",
                      (row["email"],purpose,now))
        self.audit(row["username"],"otp_requested" if purpose=="reset" else "email_verification_requested")
        return code,None

    def begin_otp(self,email,secret):
        return self._begin_code(email,secret,"resets","reset")

    def begin_email_verification(self,email,secret):
        return self._begin_code(email,secret,"email_verifications","verify")

    def cancel_otp(self,email):
        self._cancel_code(email,"resets")

    def cancel_email_verification(self,email):
        self._cancel_code(email,"email_verifications")

    def _cancel_code(self,email,table):
        row=self.user(email)
        if row:
            with self.db() as c:
                c.execute(f"DELETE FROM {table} WHERE email=?",(row["email"],))
                purpose = "reset" if table == "resets" else "verify"
                c.execute("DELETE FROM code_dispatch WHERE email=? AND purpose=?",(row["email"],purpose))

    def _verify_code(self,email,code,secret,table,purpose):
        row=self.user(email)
        if not row or not re.fullmatch(r"\d{6}",str(code)): return False
        with self.db() as c:
            rec=c.execute(f"SELECT * FROM {table} WHERE email=?",(row["email"],)).fetchone()
            if not rec or rec["expires_at"] <= int(time.time()) or rec["attempts"] >= 5:
                return False
            ok=hmac.compare_digest(otp_digest(code,rec["salt"],secret,purpose),rec["code_hash"])
            if ok:
                c.execute(f"DELETE FROM {table} WHERE email=?",(row["email"],))
                if purpose=="verify":
                    c.execute("UPDATE users SET email_verified=1,updated_at=? WHERE id=?",(utcnow(),row["id"]))
            else:
                c.execute(f"UPDATE {table} SET attempts=attempts+1 WHERE email=?",(row["email"],))
        if ok and purpose=="verify": self.audit(row["username"],"email_verified")
        return ok

    def verify_otp(self,email,code,secret):
        return self._verify_code(email,code,secret,"resets","reset")

    def verify_email(self,email,code,secret):
        return self._verify_code(email,code,secret,"email_verifications","verify")

    def recent_events(self,limit=50):
        with self.db() as c:
            return [dict(x) for x in c.execute("SELECT actor,event,occurred_at FROM audit ORDER BY id DESC LIMIT ?",(limit,))]

    def account_events(self,username,limit=30):
        with self.db() as c:
            return [dict(x) for x in c.execute("""SELECT event,occurred_at FROM audit WHERE actor=? COLLATE NOCASE
                                              ORDER BY id DESC LIMIT ?""",(username,limit))]

    def list_users(self):
        with self.db() as c:
            return [dict(x) for x in c.execute("""SELECT id,username,email,email_verified,created_at,
                previous_login,last_login,login_count FROM users ORDER BY id DESC""")]

    def delete_user(self,user_id):
        with self.db() as c:
            c.execute("DELETE FROM users WHERE id=?",(user_id,))


def issue_token(subject,secret,role="user",lifetime=7200,version=0):
    now=dt.datetime.now(UTC)
    return jwt.encode({"sub":str(subject),"role":role,"ver":int(version),"iat":now,
                       "exp":now+dt.timedelta(seconds=lifetime),"jti":secrets.token_hex(16)},
                      secret,algorithm="HS256")


def decode_token(token,secret):
    try:
        return jwt.decode(token,secret,algorithms=["HS256"],options={"require":["sub","exp","iat"]})
    except (jwt.InvalidTokenError,TypeError,ValueError):
        return None


In [ ]:
%%writefile app.py
"""Portwise | Maritime freight workspace · Milestone 1 authentication."""
import hmac
import html
import os
import re
import smtplib
import time
from email.message import EmailMessage
from pathlib import Path

import pandas as pd
import streamlit as st
from auth_core import QUESTIONS, Store, decode_token, issue_token, password_errors, valid_email, matches

st.set_page_config(page_title="Portwise | Maritime Intelligence", page_icon="⚓", layout="wide", initial_sidebar_state="collapsed")

JWT_SECRET = os.getenv("JWT_SECRET", "")
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS", "")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD", "")
ADMIN_USERNAME = os.getenv("ADMIN_USERNAME", "admin")
ADMIN_PASSWORD = os.getenv("ADMIN_PASSWORD", "")
REQUIRE_EMAIL_VERIFICATION = os.getenv("REQUIRE_EMAIL_VERIFICATION", "1").lower() not in ("0", "false", "no")
store = Store(os.getenv("DATABASE_PATH", "auth.db"))
BASE_DIR = Path(__file__).resolve().parent
ASSET_DIR = BASE_DIR / "assets"


# Streamlit serves files in ./static at /app/static when static serving is enabled.
# Keeping the artwork outside the HTML/CSS prevents multi-megabyte style tags.
LIGHT_BG = "/app/static/portwise_light_bg.png"
DARK_BG = "/app/static/portwise_dark_bg.png"

ROUTES = {"login", "signup", "forgot", "admin", "dashboard", "admin-dashboard", "demo-login", "demo-otp", "verify-email"}
requested = str(st.query_params.get("view", "login"))
if requested not in ROUTES:
    requested = "login"
for key, initial in {"view": requested, "theme": "Light", "jwt": None,
                     "reset_verified": False, "reset_identity": None,
                     "reset_method": None, "question_user": None, "flash": None,
                     "verify_email_address": None, "verify_email_until": 0}.items():
    if key not in st.session_state:
        st.session_state[key] = initial
if st.session_state.view != requested:
    st.session_state.view = requested
    st.session_state.reset_verified = False
    st.session_state.reset_identity = None
    st.session_state.question_user = None

DARK = st.session_state.theme == "Dark"
BG_URI = DARK_BG if DARK else LIGHT_BG
BG, CARD, INK, SUB, LINE, INPUT = (
    ("#04111b", "rgba(8,24,38,.64)", "#edf6fb", "#a9bfd0", "rgba(116,161,186,.26)", "#10283c") if DARK
    else ("#eef4f6", "rgba(255,255,255,.77)", "#102b37", "#5d7584", "rgba(148,183,196,.38)", "#ffffff")
)
SHELL = "rgba(255,255,255,.06)" if DARK else "rgba(255,255,255,.34)"
AUTH_BACKGROUND = f"linear-gradient({'rgba(3,15,24,.08),rgba(3,15,24,.05)' if DARK else 'rgba(255,255,255,.03),rgba(255,255,255,.08)'}), url('{BG_URI}')"
DASHBOARD_BACKGROUND = ("radial-gradient(ellipse at 80% 18%,rgba(20,144,172,.28),transparent 42%),linear-gradient(135deg,#061425,#0b304a 68%,#071322)" if DARK else "radial-gradient(ellipse at 75% 12%,rgba(125,211,225,.35),transparent 42%),linear-gradient(145deg,#eef9fc,#cde8ef 55%,#eaf7f8)")


st.markdown(f"""<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Manrope:wght@400;500;600;700;800&display=swap');
html, body, [data-testid='stApp'] {{font-family:'DM Sans',sans-serif; color:{INK};}}
[data-testid='stApp'] {{
    background-color:{BG};
    background-image:{DASHBOARD_BACKGROUND if st.session_state.view in ("dashboard", "admin-dashboard") else AUTH_BACKGROUND}!important;
    /* Keep the working wallpapers as the permanent fallback. Only their scale
       and position change gently on authentication pages. */
    background-size:cover;
    background-position:center center;
    background-attachment:fixed;
    animation:{'portwiseOceanPan 26s cubic-bezier(.45,0,.55,1) infinite alternate' if st.session_state.view not in ('dashboard','admin-dashboard') else 'none'};
}}
[data-testid='stAppViewContainer'],[data-testid='stMain'],[data-testid='stMainBlockContainer'] {{background:transparent!important;}}
[data-testid='stHeader'] {{background:transparent!important;}}
/* The atmospheric glow moves independently of the wallpaper. */
[data-testid='stApp']::after {{
    content:'';position:fixed;inset:0;z-index:0;pointer-events:none;
    background:radial-gradient(ellipse 65% 80% at 12% 78%, {'rgba(24,194,206,.13)' if DARK else 'rgba(255,255,255,.25)'}, transparent 72%),
               linear-gradient(110deg,transparent 23%,{'rgba(102,204,229,.07)' if DARK else 'rgba(255,255,255,.12)'} 51%,transparent 77%);
    background-size:145% 145%,220% 100%;
    animation:portwiseShimmer 18s ease-in-out infinite alternate;
}}
[data-testid='stAppViewContainer'] {{position:relative;z-index:1;}}
@keyframes portwiseShimmer {{
    0% {{background-position:0% 43%,0% 50%;opacity:.5;}}
    100% {{background-position:12% 53%,100% 50%;opacity:.88;}}
}}
/* A small cinematic push-in/pull-back; not a spinning or distracting effect. */
@keyframes portwiseOceanPan {{
    0% {{background-size:103% 103%; background-position:47% 50%;}}
    100% {{background-size:111% 111%; background-position:54% 48%;}}
}}
[data-testid='stHeader'] {{background:transparent;}}
.block-container {{max-width:1280px; padding-top:1.5rem; padding-bottom:1.2rem;}}
h1,h2,h3 {{font-family:'Manrope',sans-serif;letter-spacing:-.045em;color:{INK}!important;}}
p,label,[data-testid='stCaptionContainer'] {{color:{SUB};}}
[data-testid='stForm'] {{background:{CARD};border:1px solid {LINE};border-radius:22px;padding:1.15rem 1.35rem;backdrop-filter: blur(18px); box-shadow:0 20px 60px rgba(0,0,0,.10);}}
[data-testid='stVerticalBlockBorderWrapper'] > div {{background:{CARD};border-color:{LINE}!important;border-radius:20px!important;backdrop-filter: blur(18px);}}
/* Streamlit uses nested BaseWeb containers. Style the actual filled control,
   not only the outer wrapper: this prevents black fields in light mode. */
[data-testid='stTextInput'] div[data-baseweb='input'],
[data-testid='stTextInput'] div[data-baseweb='base-input'],
[data-testid='stTextInput'] div[data-baseweb='input'] > div,
[data-testid='stTextArea'] div[data-baseweb='textarea'],
[data-testid='stSelectbox'] div[data-baseweb='select'] > div {{
  background-color:{INPUT}!important;background-image:none!important;
  border-color:{LINE}!important;border-radius:12px!important;
  color:{INK}!important;
}}
[data-testid='stTextInput'] input, [data-testid='stTextArea'] textarea {{color:{INK}!important; -webkit-text-fill-color:{INK}!important; caret-color:{INK}!important; background-color:{INPUT}!important; font-weight:500!important; letter-spacing:normal!important; line-height:1.5!important;}}
[data-testid='stTextInput'] input::placeholder,[data-baseweb='input'] input::placeholder,textarea::placeholder {{color:{SUB}!important; -webkit-text-fill-color:{SUB}!important; opacity:1!important;}}
[data-baseweb='select'] > div,[data-baseweb='textarea'] > div {{background:{INPUT}!important; color:{INK}!important; border:1px solid {LINE}!important;}}
[data-baseweb='select'] [role='combobox'], [data-baseweb='select'] [role='option'] {{color:{INK}!important;}}
[data-testid='stTextInput'] label p, [data-testid='stForm'] label p, [data-testid='stSelectbox'] label p, [data-testid='stRadio'] label p {{color:{INK}!important; font-weight:600;}}
[data-testid='stButton'] button p, [data-testid='stFormSubmitButton'] button p {{color:inherit!important; font-size:14px!important; font-weight:600!important;}}
[data-testid='stWidgetLabel'] p {{color:{INK}!important;}}

[data-testid='stButton'] button,[data-testid='stFormSubmitButton'] button {{border-radius:12px;min-height:46px;border:1px solid {LINE};}}
[data-testid='stButton'] button:not([kind='primary']),[data-testid='stFormSubmitButton'] button:not([kind='primary']) {{background:{'rgba(255,255,255,.88)' if not DARK else 'rgba(12,27,44,.78)'}!important;color:{'#123448' if not DARK else '#f1f8ff'}!important;border:1px solid {LINE}!important;backdrop-filter:blur(12px);font-weight:600;}}
[data-testid='stButton'] button:not([kind='primary']) *,[data-testid='stFormSubmitButton'] button:not([kind='primary']) * {{color:{'#123448' if not DARK else '#f1f8ff'}!important;opacity:1!important;}}
[data-testid='stButton'] button[kind='primary'],[data-testid='stFormSubmitButton'] button[kind='primary'] {{background:#ff5a57;color:white;border:0;font-weight:700; box-shadow:0 12px 24px rgba(255,90,87,.18);}}
[data-testid='stButton'] button,[data-testid='stFormSubmitButton'] button {{
    transition:transform .22s ease,box-shadow .22s ease,border-color .22s ease,background-color .22s ease;
}}
[data-testid='stButton'] button:hover,[data-testid='stFormSubmitButton'] button:hover {{
    transform:translateY(-2px);box-shadow:0 10px 25px rgba(0,113,151,.16);
}}
[data-testid='stButton'] button:active,[data-testid='stFormSubmitButton'] button:active {{transform:translateY(0);}}
[data-testid='stButton'] button:focus-visible,[data-testid='stFormSubmitButton'] button:focus-visible {{
    outline:3px solid rgba(25,171,184,.55);outline-offset:3px;
}}
[data-testid='stTextInput'] div[data-baseweb='input']:focus-within,
[data-testid='stSelectbox'] div[data-baseweb='select']:focus-within {{
    border-color:#20aeb6!important;box-shadow:0 0 0 3px rgba(32,174,182,.12)!important;
}}
[data-testid='stMetric'] {{background:{CARD};border:1px solid {LINE};border-radius:16px;padding:19px; backdrop-filter: blur(16px);}}
[data-testid='stMetricValue'] {{color:{INK}!important;}}
.top-brand {{font-size:20px;font-weight:800;font-family:Manrope;color:{INK};letter-spacing:-.04em}}
.top-brand span {{color:#20aeb6;}}
.top-brand small {{font-size:11px;letter-spacing:.12em;opacity:.68;}}
.hero-wrap {{padding:1rem .4rem 1rem 0;}}
.kicker {{font-size:11px;font-weight:800;text-transform:uppercase;letter-spacing:.18em;color:#28aeb7;margin-bottom:12px;}}
.headline {{font-family:Manrope;font-size:clamp(42px,5vw,76px);letter-spacing:-.06em;font-weight:800;line-height:1.04;color:{INK};margin:12px 0 18px;}}
.headline em {{color:#24aeb4;font-style:normal;}}
.intro-copy {{max-width:520px;font-size:16px;line-height:1.8;color:{SUB};}}
.pill {{display:inline-block;border:1px solid {LINE};background:{SHELL};backdrop-filter: blur(12px);border-radius:999px;padding:9px 14px;font-size:11px;font-weight:700;letter-spacing:.075em;color:{SUB};margin:8px 8px 0 0;}}
.hero-card {{margin-top:24px; max-width:560px; border:1px solid {LINE}; border-radius:24px; background:{CARD}; backdrop-filter: blur(18px); padding:22px 24px; box-shadow:0 24px 70px rgba(0,0,0,.08);}}
.hero-card h4 {{font-family:Manrope; font-size:20px; margin:0 0 10px; color:{INK};}}
.hero-card p {{font-size:14px; line-height:1.8; color:{SUB}; margin:0 0 16px;}}
.signal-grid {{display:grid; grid-template-columns:repeat(2,minmax(0,1fr)); gap:12px;}}
.signal-tile {{border:1px solid {LINE}; border-radius:16px; padding:14px 14px 12px; background:transparent;}}
.signal-tile strong {{display:block; font-family:Manrope; color:{INK}; font-size:15px; margin-bottom:3px;}}
.signal-tile span {{display:block; color:{SUB}; font-size:12px; line-height:1.55;}}
.side-note {{max-width:560px; margin-top:14px; color:{SUB}; font-size:12px; line-height:1.7;}}
.flow-card {{border:1px solid {LINE};background:{CARD};backdrop-filter: blur(16px);border-radius:16px;padding:19px 20px;margin-bottom:13px;}}
.flow-card b {{color:{INK};font-family:Manrope;}}
.flow-card p {{font-size:13px;margin:6px 0 0;}}
.step-num {{display:inline-block;background:#087f8a;color:white;border-radius:50%;width:25px;height:25px;text-align:center;line-height:25px;margin-right:10px;font-weight:700;}}

.otp-status {{display:flex;justify-content:space-between;align-items:center;background:{CARD};border:1px solid {LINE};border-radius:14px;padding:16px 19px;margin:8px 0 14px;}}
.otp-status strong {{font-size:25px;font-family:Manrope;color:#16afb8;letter-spacing:.06em;}}
.dashboard-hero {{background:{CARD};backdrop-filter:blur(16px);border:1px solid {LINE};border-radius:24px;padding:30px;margin:8px 0 20px;box-shadow:0 20px 55px rgba(0,0,0,.07);animation:portwiseEnter .8s ease both;}}
.dashboard-hero h2 {{margin:9px 0;font-size:clamp(27px,3vw,42px);}}
.footer-note {{margin-top:36px;text-align:center;color:{SUB};font-size:12px;}}
/* Entrance hierarchy: branding -> message -> form. Reruns keep their fields. */
.top-brand {{animation:portwiseFadeIn .65s ease-out both;}}
.hero-wrap {{animation:portwiseEnter .92s cubic-bezier(.22,.7,.28,1) .1s both;}}
.kicker {{animation:portwiseFadeIn .85s ease-out .18s both;}}
.headline em {{display:inline-block; animation:portwiseTint 6s ease-in-out infinite alternate;}}
[data-testid='stForm'] {{animation:portwiseEnter .72s cubic-bezier(.22,.7,.28,1) .16s both;}}
.dashboard-hero {{animation:portwiseEnter .75s cubic-bezier(.22,.7,.28,1) both;}}
[data-testid='stMetric'] {{transition:transform .25s ease,box-shadow .25s ease;}}
[data-testid='stMetric']:hover {{transform:translateY(-2px);box-shadow:0 14px 32px rgba(2,33,46,.09);}}
@keyframes portwiseEnter {{
  from {{opacity:0;transform:translate3d(0,18px,0);}}
  to {{opacity:1;transform:translate3d(0,0,0);}}
}}
@keyframes portwiseFadeIn {{from {{opacity:0;}} to {{opacity:1;}}}}
@keyframes portwiseTint {{
  from {{filter:brightness(1);}}
  to {{filter:brightness(1.17);}}
}}
@media (prefers-reduced-motion: reduce) {{
  [data-testid='stApp'],[data-testid='stApp']::after,.top-brand,.hero-wrap,.kicker,.headline em,
  [data-testid='stForm'],.dashboard-hero,[data-testid='stMetric'],
  [data-testid='stButton'] button,[data-testid='stFormSubmitButton'] button {{
     animation:none!important;transition:none!important;transform:none!important;
  }}
}}
@media (max-width: 700px) {{
    /* Don't crop or stretch the art on narrow/mobile screens. */
    [data-testid='stApp'] {{background-size:cover!important;animation:none!important;}}
}}
@media (max-width: 992px) {{
  .block-container {{padding-top:1rem;}}
  .hero-wrap {{padding-right:0;}}
  .headline {{font-size:clamp(34px,8vw,54px);}}
}}
</style>""", unsafe_allow_html=True)


def navigate(view):
    st.session_state.view = view
    st.query_params["view"] = view
    st.rerun()


def flash():
    if st.session_state.flash:
        st.success(st.session_state.flash)
        st.session_state.flash = None


def current_user():
    token = st.session_state.jwt
    payload = decode_token(token, JWT_SECRET) if token else None
    if not payload:
        return None
    if payload.get("role") == "admin":
        return {"role": "admin", "username": ADMIN_USERNAME} if payload.get("sub") == ADMIN_USERNAME else None
    row = store.user(payload.get("sub", ""))
    return (dict(row) | {"role": "user"}) if row and row["email_verified"] and payload.get("ver") == row["session_version"] and payload.get("role") == "user" else None


def log_out():
    account = current_user()
    if account:
        store.audit(account["username"], "logout")
    st.session_state.jwt = None
    st.session_state.reset_verified = False
    navigate("login")


def password_ok(password, confirmation):
    missing = password_errors(password)
    if missing:
        st.error("Password must contain " + ", ".join(missing) + ".")
        return False
    if password != confirmation:
        st.error("Passwords do not match.")
        return False
    return True


def send_code(recipient, code, purpose="reset"):
    if not EMAIL_ADDRESS or not EMAIL_PASSWORD:
        return False, "Gmail SMTP sending credentials have not been configured."
    mail = EmailMessage()
    mail["From"] = EMAIL_ADDRESS
    mail["To"] = recipient
    title = "Verify your PORTWISE email" if purpose == "signup" else "PORTWISE password recovery"
    mail["Subject"] = title
    mail.set_content(f"Your PORTWISE six-digit code is {code}. It expires in five minutes. If you didn't request it, ignore this email.")
    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=20) as smtp:
            smtp.login(EMAIL_ADDRESS.strip(), EMAIL_PASSWORD.replace(" ", ""))
            refused = smtp.send_message(mail)
            if refused:
                return False, "The email service refused the recipient. Check the address and try again."
        return True, None
    except smtplib.SMTPAuthenticationError:
        return False, "Gmail rejected the sender login. Verify the Gmail App Password in Colab Secrets."
    except (smtplib.SMTPException, OSError):
        return False, "Email sending failed. Check the sender settings and SMTP connectivity in Colab."


def topbar():
    left, right = st.columns([5, 1], vertical_alignment="center")
    with left:
        st.markdown('<div class="top-brand">◈ PORT<span>WISE</span> <small>/ MARITIME INTELLIGENCE</small></div>', unsafe_allow_html=True)
    with right:
        if st.button("☀ Light" if DARK else "☾ Dark", use_container_width=True):
            st.session_state.theme = "Light" if DARK else "Dark"
            st.rerun()


def left_panel():
    st.markdown(
        '<div class="hero-wrap">'
        '<div class="kicker">Agentic AI / Freight intelligence</div>'
        '<div class="headline">Navigate freight.<br><em>Intelligently.</em></div>'
        '<p class="intro-copy">A secure entry point for the next generation of maritime freight pricing and route optimization. Connecting people, ports and decisions.</p>'
        '</div>',
        unsafe_allow_html=True,
    )


def login():
    st.markdown("### Sign in to Portwise")
    st.caption("Welcome back. Enter your credentials to continue.")
    flash()
    with st.form("login_form"):
        identity = st.text_input("Username or email", placeholder="Your username or work email")
        password = st.text_input("Password", type="password", placeholder="Your password")
        submit = st.form_submit_button("Sign in →", type="primary", use_container_width=True)
    if submit:
        if not identity.strip() or not password:
            st.error("Both fields are required.")
        else:
            account, error = store.authenticate(identity, password)
            if error:
                st.error(error)
            else:
                st.session_state.jwt = issue_token(account["email"], JWT_SECRET, version=account["session_version"])
                navigate("dashboard")
    a, b = st.columns(2)
    with a:
        if st.button("Create account", use_container_width=True):
            navigate("signup")
    with b:
        if st.button("Forgot password?", use_container_width=True):
            navigate("forgot")
    st.divider()
    if st.button("Admin sign in", use_container_width=True):
        navigate("admin")
    with st.expander("Account help"):
        if st.button("Verify a registered email address", use_container_width=True):
            navigate("verify-email")
        if st.button("Login demonstration", use_container_width=True):
            navigate("demo-login")


def signup():
    st.markdown("### Create an account")
    st.caption("Set up your secure Portwise profile.")
    with st.form("signup_form"):
        username = st.text_input("Username")
        email = st.text_input("Email address")
        a, b = st.columns(2)
        with a:
            password = st.text_input("Password", type="password")
        with b:
            confirm = st.text_input("Confirm password", type="password")
        question = st.selectbox("Security question", QUESTIONS, index=None, placeholder="Choose a question")
        answer = st.text_input("Security answer", type="password")
        st.caption("Password: 8+ characters, uppercase, lowercase, digit and special character.")
        submit = st.form_submit_button("Create account →", type="primary", use_container_width=True)
    if submit:
        if not all((username.strip(), email.strip(), password, confirm, question, answer.strip())):
            st.error("Complete all required fields.")
        elif not valid_email(email):
            st.error("Enter a valid email, such as ab@cd.ef.")
        elif password_ok(password, confirm):
            if REQUIRE_EMAIL_VERIFICATION and not (EMAIL_ADDRESS and EMAIL_PASSWORD):
                st.error("Email verification is enabled but the sending account has not been configured. Ask the administrator to configure the email service.")
                return
            ok, message = store.signup(username, email, password, question, answer,
                                       require_verification=REQUIRE_EMAIL_VERIFICATION)
            if ok:
                if REQUIRE_EMAIL_VERIFICATION:
                    st.session_state.verify_email_address = email.strip().lower()
                    # Attempt delivery; resend remains available if SMTP cannot accept.
                    st.session_state.flash = message
                    navigate("verify-email")
                else:
                    st.session_state.flash = message
                    navigate("login")
            else:
                st.error(message)
    if st.button("← Back to sign in"):
        navigate("login")


@st.fragment(run_every="1s")
def otp_timer():
    remaining = max(0, int(st.session_state.get("otp_expires_at", 0) - time.time()))
    mins, secs = divmod(remaining, 60)
    if remaining:
        st.markdown(f'<div class="otp-status"><span>⏱ Code expires in</span><strong>{mins:02d}:{secs:02d}</strong></div>', unsafe_allow_html=True)
    else:
        st.warning("Your verification code has expired. You can request another one.")
    resend = st.button("Resend verification code", disabled=remaining > 0, use_container_width=True, key="resend_otp")
    if resend:
        address = st.session_state.get("reset_identity")
        if address:
            deliver_otp(address)
            st.rerun()


def deliver_otp(address):
    code, message = store.begin_otp(address, JWT_SECRET)
    if not code:
        st.info(message or "If the address is registered, check your inbox. If no verification form appears, confirm this account was created in the current Colab database.")
        return
    sent, issue = send_code(address, code)
    if not sent:
        store.cancel_otp(address)
        st.error(issue or "Email delivery could not be accepted by the sending server.")
        return
    st.session_state.reset_identity = address
    st.session_state.otp_expires_at = time.time() + 300
    st.success("Code request accepted by the email server. Check Inbox, Promotions and Spam. Delivery can take a moment.")



def send_signup_verification(address):
    code, note = store.begin_email_verification(address, JWT_SECRET)
    if not code:
        st.info(note or "If your account is eligible, check your email.")
        return False
    accepted, problem = send_code(address, code, purpose="signup")
    if not accepted:
        store.cancel_email_verification(address)
        st.error(problem or "The email server did not accept the verification message.")
        return False
    st.session_state.verify_email_address = address
    st.session_state.verify_email_until = time.time() + 300
    st.success("Verification email accepted by the sending server. Check your inbox and spam folder.")
    return True


@st.fragment(run_every="1s")
def signup_verification_countdown():
    remaining = max(0, int(st.session_state.get("verify_email_until", 0) - time.time()))
    if remaining:
        mins, secs = divmod(remaining, 60)
        st.markdown(f'<div class="otp-status"><span>⏱ Code expires in</span><strong>{mins:02d}:{secs:02d}</strong></div>', unsafe_allow_html=True)
    else:
        st.caption("You can request a new verification code.")
    if st.button("Resend email verification", disabled=remaining > 0, key="resend_signup_verify"):
        if st.session_state.verify_email_address:
            send_signup_verification(st.session_state.verify_email_address)
            st.rerun()


def verify_email_page():
    st.markdown("### Verify your email")
    st.caption("Confirm your email address before signing in to PORTWISE.")
    flash()
    with st.form("email_verification_request"):
        address = st.text_input("Registered email address", value=st.session_state.verify_email_address or "")
        request = st.form_submit_button("Send verification code", type="primary", use_container_width=True)
    if request:
        if not valid_email(address):
            st.error("Enter a valid email address.")
        elif not EMAIL_ADDRESS or not EMAIL_PASSWORD:
            st.error("Email delivery is not configured yet.")
        else:
            send_signup_verification(address.strip().lower())
    if st.session_state.verify_email_address and st.session_state.verify_email_until:
        signup_verification_countdown()
        with st.form("verify_signup_code"):
            code = st.text_input("Six-digit code", max_chars=6, placeholder="000000")
            confirm = st.form_submit_button("Verify email →", type="primary", use_container_width=True)
        if confirm:
            if store.verify_email(st.session_state.verify_email_address, code.strip(), JWT_SECRET):
                st.session_state.verify_email_until = 0
                st.session_state.verify_email_address = None
                st.session_state.flash = "Email verified. Sign in to continue."
                navigate("login")
            else:
                st.error("Code invalid or expired. Check your message or request a new code after the timer ends.")
    if st.button("← Return to login", key="verify_back"):
        navigate("login")


def forgot():
    st.markdown("### Recover your account")
    st.caption("Choose email verification or your saved security question. Codes are six digits and valid for five minutes.")
    method = st.radio("Recovery method", ["Email OTP", "Security question"], horizontal=True)
    kind = "otp" if method == "Email OTP" else "question"
    if st.session_state.reset_method != kind:
        st.session_state.reset_method = kind
        st.session_state.reset_verified = False
        st.session_state.reset_identity = None
        st.session_state.question_user = None
        st.session_state.otp_expires_at = 0
    if st.session_state.reset_verified:
        st.success("Identity verified. Set a new password.")
        with st.form("new_password_form"):
            pw = st.text_input("New password", type="password")
            confirm = st.text_input("Confirm new password", type="password")
            submit = st.form_submit_button("Reset password →", type="primary", use_container_width=True)
        if submit and password_ok(pw, confirm):
            ok, message = store.reset_password(st.session_state.reset_identity, pw)
            if ok:
                st.session_state.reset_verified = False
                st.session_state.reset_identity = None
                st.session_state.question_user = None
                st.session_state.flash = message
                navigate("login")
            else:
                st.error(message)
    elif method == "Email OTP":
        with st.form("send_otp_form"):
            address = st.text_input("Registered email", placeholder="you@gmail.com or you@outlook.com")
            send = st.form_submit_button("Send verification code", type="primary", use_container_width=True)
        if send:
            if not valid_email(address):
                st.error("Enter a valid email address.")
            elif not EMAIL_ADDRESS or not EMAIL_PASSWORD:
                st.error("Email OTP requires EMAIL_ADDRESS and EMAIL_PASSWORD in the server configuration.")
            else:
                deliver_otp(address.strip().lower())
        if st.session_state.reset_identity:
            st.info("A six-digit code was requested for " + st.session_state.reset_identity + ". If it does not arrive, check Spam or Promotions and verify your sender configuration.")
            st.markdown("#### Verify your email")
            otp_timer()
            with st.form("verify_otp_form"):
                code = st.text_input("Six-digit OTP", max_chars=6, placeholder="Enter the code from your email")
                verify = st.form_submit_button("Verify OTP →", type="primary", use_container_width=True)
            if verify:
                if not re.fullmatch(r"[0-9]{6}", code.strip()):
                    st.error("Enter the complete six-digit code.")
                elif store.verify_otp(st.session_state.reset_identity, code.strip(), JWT_SECRET):
                    st.session_state.reset_verified = True
                    st.rerun()
                else:
                    st.error("The code was incorrect, expired, or exceeded the maximum attempts. You can request another code after the resend delay.")
    else:
        with st.form("find_question_form"):
            username = st.text_input("Registered username")
            find = st.form_submit_button("Find security question", type="primary", use_container_width=True)
        if find:
            question = store.security_question(username)
            if question:
                st.session_state.question_user = username.strip()
            else:
                st.error("Unable to verify that username.")
        user = st.session_state.question_user
        if user:
            question = store.security_question(user)
            if question:
                st.info(question)
                with st.form("answer_question_form"):
                    answer = st.text_input("Your answer", type="password")
                    verify = st.form_submit_button("Verify answer →", type="primary", use_container_width=True)
                if verify:
                    if store.verify_answer(user, answer):
                        st.session_state.reset_identity = store.user(user)["email"]
                        st.session_state.reset_verified = True
                        st.rerun()
                    else:
                        st.error("Incorrect answer or too many attempts.")
    a, b = st.columns(2)
    with a:
        if st.button("← Sign in", use_container_width=True):
            navigate("login")
    with b:
        if st.button("OTP walkthrough", use_container_width=True):
            navigate("demo-otp")


def admin_login():
    st.markdown("### Administration")
    st.caption("Restricted access for milestone reviewers and administrators.")
    if not ADMIN_PASSWORD:
        st.warning("Configure ADMIN_PASSWORD to enable administrator access.")
    with st.form("admin_login_form"):
        username = st.text_input("Admin username")
        password = st.text_input("Admin password", type="password")
        submit = st.form_submit_button("Open admin dashboard →", type="primary", use_container_width=True)
    if submit:
        if not ADMIN_PASSWORD:
            st.error("Admin login is not yet configured.")
        elif store.limited("admin-login"):
            st.error("Too many administrator sign-in attempts. Try again in 15 minutes.")
        elif hmac.compare_digest(username, ADMIN_USERNAME) and hmac.compare_digest(password, ADMIN_PASSWORD):
            store.clear_limit("admin-login")
            st.session_state.jwt = issue_token(ADMIN_USERNAME, JWT_SECRET, role="admin", lifetime=3600)
            store.audit(ADMIN_USERNAME, "admin_login")
            navigate("admin-dashboard")
        else:
            store.failure("admin-login")
            store.audit("admin", "admin_login_failed")
            st.error("Invalid administrator credentials.")
    if st.button("← Back to sign in"):
        navigate("login")


def user_dashboard(account):
    top, out = st.columns([5, 1], vertical_alignment="center")
    with top:
        st.markdown('<div class="kicker">PORTWISE / Your maritime workspace</div>', unsafe_allow_html=True)
    with out:
        if st.button("Log out", use_container_width=True):
            log_out()
    username = html.escape(account["username"])
    last_visit = account["previous_login"]
    visit_note = ("Your previous successful sign-in was " + html.escape(last_visit.replace("T", " ")[:16]) + " UTC."
                  if last_visit else "Welcome to your PORTWISE workspace. This is your first recorded sign-in.")
    st.markdown(f'<div class="dashboard-hero"><div class="kicker">WELCOME BACK</div><h2>Good to see you, {username}.</h2><p>{visit_note}</p></div>', unsafe_allow_html=True)
    events = store.account_events(account["username"], 50)
    a, b, c = st.columns(3)
    a.metric("Account status", "Active")
    b.metric("Successful logins", account["login_count"])
    c.metric("Recent account events", len(events))
    activity, profile, security = st.tabs(["Recent activity", "Your profile", "Security settings"])
    with activity:
        if events:
            st.dataframe(pd.DataFrame(events)[["event", "occurred_at"]], use_container_width=True, hide_index=True)
        else:
            st.info("No recent account activity has been recorded yet.")
        st.caption("This is actual recorded account activity—not simulated shipping data.")
    with profile:
        st.write("**Username:**", account["username"])
        st.write("**Email:**", account["email"])
        st.write("**Member since:**", account["created_at"][:10])
    with security:
        st.subheader("Change password")
        st.caption("Your current password is required. Changing it signs you out of all existing sessions.")
        with st.form("change_password_form"):
            current = st.text_input("Current password", type="password")
            new = st.text_input("New password", type="password")
            confirm = st.text_input("Confirm new password", type="password")
            save = st.form_submit_button("Update password", type="primary")
        if save:
            if not current:
                st.error("Enter your current password.")
            elif password_ok(new, confirm):
                ok, message = store.change_password(account["email"], current, new)
                if ok:
                    st.session_state.jwt = None
                    st.session_state.flash = "Password changed. Sign in again with the new password."
                    navigate("login")
                else:
                    st.error(message)
    st.caption("Freight pricing and route optimization will be added in later milestones.")


def admin_dashboard():
    top, out = st.columns([5, 1], vertical_alignment="center")
    with top:
        st.markdown('<div class="kicker">Restricted access / administrator</div>', unsafe_allow_html=True)
        st.title("Platform administration")
    with out:
        if st.button("Log out", use_container_width=True):
            log_out()
    users = store.list_users()
    events = store.recent_events(50)
    a, b, c = st.columns(3)
    a.metric("Registered members", len(users))
    b.metric("Total logins", sum(item["login_count"] for item in users))
    c.metric("Recent audit events", len(events))
    members, activity = st.tabs(["Registered users", "Activity history"])
    with members:
        query = st.text_input("Search users", placeholder="Username or email")
        matching = [u for u in users if query.casefold() in (u["username"] + " " + u["email"]).casefold()]
        if matching:
            st.dataframe(pd.DataFrame(matching)[["username", "email", "email_verified", "created_at", "last_login", "login_count"]], use_container_width=True, hide_index=True)
        else:
            st.info("No users found.")
    with activity:
        if events:
            st.dataframe(pd.DataFrame(events), use_container_width=True, hide_index=True)
        else:
            st.info("No activity yet.")


def walkthrough(which):
    if which == "login":
        st.markdown('<div class="kicker">Demonstration 01 / account access</div>', unsafe_allow_html=True)
        st.title("Login & signup walkthrough")
        steps = [("01", "Create an account", "Provide username, email, strong password and security question."),
                 ("02", "Sign in", "Enter either your registered username or email plus your password."),
                 ("03", "JWT session", "A signed token is issued only after valid authentication."),
                 ("04", "Dashboard & logout", "The protected dashboard opens; logout clears the session.")]
    else:
        st.markdown('<div class="kicker">Demonstration 02 / account recovery</div>', unsafe_allow_html=True)
        st.title("Email OTP & recovery walkthrough")
        steps = [("01", "Register your email", "Create an account with an email inbox you can access."),
                 ("02", "Request the code", "On Forgot Password, choose Email OTP and supply that email."),
                 ("03", "Verify", "Gmail SMTP sends a six-digit code; it expires in five minutes."),
                 ("04", "Choose a new password", "After verification, reset your password and log in again.")]
    st.caption("This is a guided demonstration page. Use the buttons below to test the actual forms.")
    for number, title, description in steps:
        st.markdown(f'<div class="flow-card"><span class="step-num">{number[-1]}</span><b>{html.escape(title)}</b><p>{html.escape(description)}</p></div>', unsafe_allow_html=True)
    a, b = st.columns(2)
    with a:
        if st.button("Start live " + ("login" if which == "login" else "OTP") + " demo →", type="primary", use_container_width=True):
            navigate("login" if which == "login" else "forgot")
    with b:
        if st.button("View other walkthrough", use_container_width=True):
            navigate("demo-otp" if which == "login" else "demo-login")


if len(JWT_SECRET) < 32:
    st.error("JWT signing key is not configured. Run the Colab configuration cell first.")
    st.stop()

topbar()
viewer = current_user()
if viewer and viewer["role"] == "admin":
    if st.session_state.view != "admin-dashboard":
        navigate("admin-dashboard")
    admin_dashboard()
elif viewer:
    if st.session_state.view != "dashboard":
        navigate("dashboard")
    user_dashboard(viewer)
else:
    if st.session_state.jwt:
        st.session_state.jwt = None
    left, right = st.columns([1.05, .95], gap="large", vertical_alignment="center")
    with left:
        left_panel()
    with right:
        route = st.session_state.view
        if route == "signup":
            signup()
        elif route == "forgot":
            forgot()
        elif route == "verify-email":
            verify_email_page()
        elif route == "admin":
            admin_login()
        elif route == "demo-login":
            walkthrough("login")
        elif route == "demo-otp":
            walkthrough("otp")
        else:
            login()
build_label = ' · <strong>Build: PORTWISE-FINAL-01</strong>' if os.getenv('SHOW_BUILD_ID', '0') == '1' else ''
st.markdown('<div class="footer-note">PORTWISE · Agentic AI for Maritime Freight Pricing and Route Optimization' + build_label + '</div>', unsafe_allow_html=True)


## 4 · Install the original approved maritime backgrounds

This code cell contains the two unchanged PNG files, compressed into the notebook so you only have to upload **one** `.ipynb` file. Its source is collapsed to prevent long encoded image text from filling your screen.

In [ ]:
#@title Restore PORTWISE maritime artwork (collapsed) { display-mode: "form" }
import io, base64, zipfile
from pathlib import Path
Path('static').mkdir(exist_ok=True)
image_package = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(image_package))) as z:
    z.extractall('static')
del image_package
print('Approved light and dark maritime backgrounds installed.')


## 5 · Write Streamlit settings and run the test suite

In [ ]:
%%writefile README.md
# PORTWISE — Milestone 1

This milestone builds the user authentication module for PORTWISE, a maritime freight project. Freight pricing and route optimization are planned for later milestones. The current app covers account creation, sign-in, account recovery, and separate user and administrator dashboards.

## Features

- Signup with a unique username and email, password confirmation, and a saved security question.
- Login with username or email; the same error is shown for an incorrect username or password.
- Signed JWT sessions, protected dashboards, logout, and session invalidation after password changes.
- Password recovery using a security answer or a six-digit email OTP valid for five minutes.
- Separate administrator sign-in and a registered-user directory without passwords or hashes.
- Hashed passwords and recovery answers, OTP attempt limits, login throttling, and account activity records.
- Light and dark maritime backgrounds. Email verification at signup is an extra feature.

## Stack

Python, Streamlit, SQLite, bcrypt, PyJWT, pandas, Gmail SMTP, and pyngrok for the Colab tunnel.

## Run in Google Colab

1. Open `PORTWISE_Final_All_in_One_Colab.ipynb` in Colab.
2. In the Secrets panel, add `JWT_SECRET` (a random key of at least 32 characters), `NGROK_AUTHTOKEN`, `EMAIL_ADDRESS`, `EMAIL_PASSWORD`, and `ADMIN_PASSWORD` (at least 12 characters). Enable notebook access for each. `ADMIN_USERNAME` is optional and defaults to `admin`.
3. Use a Gmail App Password for `EMAIL_PASSWORD`; enable Google 2-Step Verification first. The email address is the sending account, not the recipient.
4. Run the cells from top to bottom. The notebook writes the app, installs its artwork, runs tests, checks SMTP login, and opens a public ngrok link.
5. Create an account, open the email verification page, request a code, verify the email, and sign in. Try both recovery methods and the separate admin sign-in.
6. Keep Colab connected during the demo. The SQLite database is temporary and can disappear when the runtime is deleted.
7. Before submitting an executed copy, clear all outputs and check that credentials and tunnel URLs are absent.

All four secrets named in the brief should be configured explicitly even though the notebook can generate a temporary JWT key. No real credentials are included.

## Run locally

Use Python 3.12. From this folder:

```bash
python -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
cp .env.example .env
# Edit .env privately, then load it:
set -a
source .env
set +a
streamlit run app.py
```

On Windows, activate `.venv` and set the same environment variables in PowerShell instead. Open http://localhost:8501. `DATABASE_PATH=./portwise_auth.db` is suitable locally. The notebook uses its own Colab path.

## Tests

```bash
python -m unittest discover -s tests -v
python -m py_compile app.py auth_core.py
```

These tests do not establish real email delivery or a successful Colab/ngrok run. Those require the owner's configured accounts.

## Files

`app.py` contains the interface and routing; `auth_core.py` handles accounts, OTPs, and JWTs. `static/` contains the backgrounds, `.streamlit/` the configuration, and `tests/` the checks. The notebook is self-contained and recreates the runnable files. Docker files are optional; for Docker, set `DATABASE_PATH=/data/portwise_auth.db` in your private `.env` so the supplied volume retains the database.

## Submission status

Extract this ZIP and upload the **Milestone1 folder and its contents** into your personal Infosys repository. Do not upload the ZIP alone. This package contains the cleaned final notebook and extracted source, not the two older reference notebooks. See `REVIEW.md` for the comparison and remaining checks.

The brief asks for admin credentials directly in code but later asks for the admin password to be removed. This submission keeps the admin separate from signup users and reads its password from Secrets. Confirm that interpretation with the mentor if literal hard-coded credentials are expected.

## Screenshots

The following are the required screenshot destinations. Images are pending; add your own live captures before submission. Once files are added, the links will display them. Use a test account and redact personal addresses, passwords, OTP digits, tokens, and tunnel URLs.

| Capture | Image |
|---|---|
| Login | ![Login](screenshots/login.png) |
| Signup | ![Signup](screenshots/signup.png) |
| Forgot Password — security question | ![Security question recovery](screenshots/forgot_security_question.png) |
| Forgot Password — OTP | ![OTP recovery](screenshots/forgot_otp.png) |
| OTP email received | ![OTP email](screenshots/otp_email.png) |
| User dashboard | ![User dashboard](screenshots/user_dashboard.png) |
| Admin dashboard | ![Admin dashboard](screenshots/admin_dashboard.png) |


In [ ]:
%%writefile Dockerfile
FROM python:3.12-slim
WORKDIR /app
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1 DATABASE_PATH=/data/portwise_auth.db
COPY requirements.txt ./
RUN pip install --no-cache-dir -r requirements.txt \
    && useradd --create-home --uid 10001 portwise \
    && mkdir -p /data && chown portwise:portwise /data
COPY app.py auth_core.py ./
COPY static ./static
COPY .streamlit ./.streamlit
RUN chown -R portwise:portwise /app
USER portwise
EXPOSE 8501
HEALTHCHECK --interval=30s --timeout=5s --start-period=30s --retries=3 CMD python -c "import urllib.request; urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health', timeout=4)" || exit 1
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0"]


In [ ]:
%%writefile compose.yaml
services:
  portwise:
    build: .
    restart: unless-stopped
    env_file: .env
    ports:
      # Intentionally localhost-only. Use an HTTPS reverse proxy for public access.
      - "127.0.0.1:8501:8501"
    volumes:
      - portwise_data:/data
    security_opt:
      - no-new-privileges:true

volumes:
  portwise_data:


In [ ]:
%%writefile requirements.txt
streamlit>=1.39,<2
bcrypt>=4.1,<6
PyJWT>=2.8,<3
pandas>=2,<3


In [ ]:
%%writefile .env.example
# Copy to .env for local deployment only. NEVER commit .env.
# Generate a key with: python -c "import secrets; print(secrets.token_urlsafe(48))"
JWT_SECRET=REPLACE_WITH_A_RANDOM_SECRET_OF_AT_LEAST_32_CHARACTERS
ADMIN_USERNAME=admin
ADMIN_PASSWORD=REPLACE_WITH_A_LONG_RANDOM_ADMIN_PASSWORD
EMAIL_ADDRESS=your-sender@gmail.com
EMAIL_PASSWORD=YOUR_GOOGLE_APP_PASSWORD
DATABASE_PATH=./portwise_auth.db
# Set to 0 only for the milestone demonstration when SMTP isn't configured.
REQUIRE_EMAIL_VERIFICATION=1


In [ ]:
%%writefile .gitignore
.env
*.db
*.db-shm
*.db-wal
__pycache__/
.pytest_cache/
.ipynb_checkpoints/
.venv/
*.log
*.pem
*.key


In [ ]:
from pathlib import Path
Path('.streamlit').mkdir(exist_ok=True)
Path('tests').mkdir(exist_ok=True)
Path('screenshots').mkdir(exist_ok=True)
Path('.streamlit/config.toml').write_text('[theme]\nbase = "light"\nprimaryColor = "#16A3B1"\nbackgroundColor = "#F2F8FB"\ntextColor = "#102B37"\n\n[server]\nheadless = true\nenableStaticServing = true\nenableXsrfProtection = true\nenableCORS = true\nmaxUploadSize = 10\n\n[browser]\ngatherUsageStats = false\n')
print('Streamlit theme, static-serving and security settings written.')

In [ ]:
%%writefile tests/test_assets.py
import hashlib
import unittest
from pathlib import Path

HERE = Path(__file__).resolve().parents[1]

class VisualAssetTests(unittest.TestCase):
    def test_original_wallpapers_unchanged(self):
        signatures = {
            'portwise_light_bg.png':'73485e3a2d4bf490f08997a6d346f6a0bf96fca9d6fd91b843c360556bce1d9b',
            'portwise_dark_bg.png':'7c8d5533aaf447da46936547cde3e09caaf5dc45666749d925a0d5e5f4f28313',
        }
        for name, expected in signatures.items():
            data=(HERE/'static'/name).read_bytes()
            self.assertTrue(data.startswith(b'\x89PNG\r\n\x1a\n'))
            self.assertEqual(hashlib.sha256(data).hexdigest(), expected)

    def test_theme_and_motion_present(self):
        text=(HERE/'app.py').read_text()
        for snippet in ('portwiseOceanPan','portwiseShimmer','prefers-reduced-motion',
                        '"theme": "Light"','"verify-email"','otp_timer','verify_email_page'):
            self.assertIn(snippet,text)


if __name__=='__main__':
    unittest.main()


In [ ]:
%%writefile tests/test_auth_core.py
import tempfile
import time
import unittest
from pathlib import Path

from auth_core import Store, issue_token, decode_token, password_errors, valid_email

QUESTION = 'What was the name of your first school?'
SECRET = 'a' * 48


class AuthTests(unittest.TestCase):
    def setUp(self):
        self.tmp = tempfile.TemporaryDirectory()
        self.db = Store(Path(self.tmp.name) / 'test.db')
        ok, msg = self.db.signup('alex', 'al@cd.ef', 'StrongPwd1!', QUESTION, 'Sample School')
        self.assertTrue(ok, msg)

    def tearDown(self):
        self.tmp.cleanup()

    def test_signup_validation_and_duplicates(self):
        self.assertTrue(valid_email('ab@cd.ef'))
        self.assertTrue(valid_email('an@outlook.com'))
        self.assertFalse(valid_email('a@b.c'))
        self.assertTrue(password_errors('weak'))
        self.assertFalse(self.db.signup('alex', 'xy@cd.ef', 'StrongPwd1!', QUESTION, 'A')[0])
        self.assertFalse(self.db.signup('second', 'al@cd.ef', 'StrongPwd1!', QUESTION, 'A')[0])

    def test_login_generic_errors_and_activity(self):
        self.assertIsNotNone(self.db.authenticate('alex', 'StrongPwd1!')[0])
        missing, err1 = self.db.authenticate('missing', 'wrong')
        incorrect, err2 = self.db.authenticate('alex', 'wrong')
        self.assertIsNone(missing)
        self.assertIsNone(incorrect)
        self.assertEqual(err1, err2)
        self.assertGreaterEqual(len(self.db.account_events('alex')), 2)

    def test_limit(self):
        for _ in range(5):
            self.db.failure('admin-login')
        self.assertTrue(self.db.limited('admin-login'))
        self.db.clear_limit('admin-login')
        self.assertFalse(self.db.limited('admin-login'))

    def test_question_and_reset_revokes_sessions(self):
        self.assertTrue(self.db.verify_answer('alex', 'sample school'))
        account = self.db.user('alex')
        version = account['session_version']
        self.assertTrue(self.db.reset_password('al@cd.ef', 'NewStrongPwd2!')[0])
        self.assertEqual(self.db.user('alex')['session_version'], version+1)
        self.assertIsNotNone(self.db.authenticate('alex', 'NewStrongPwd2!')[0])
        self.assertIsNone(self.db.authenticate('alex', 'StrongPwd1!')[0])

    def test_otp_reset_and_expiration(self):
        code, error = self.db.begin_otp('al@cd.ef', SECRET)
        self.assertFalse(error)
        self.assertRegex(code, r'^\d{6}$')
        self.assertFalse(self.db.verify_otp('al@cd.ef', '00000x', SECRET))
        self.assertTrue(self.db.verify_otp('al@cd.ef', code, SECRET))
        self.assertFalse(self.db.verify_otp('al@cd.ef', code, SECRET))
        other, note = self.db.begin_otp('al@cd.ef', SECRET)
        self.assertIsNone(other)
        self.assertIn('five minutes', note)
        self.db.cancel_otp('al@cd.ef')
        code, _ = self.db.begin_otp('al@cd.ef', SECRET)
        with self.db.db() as c:
            c.execute('UPDATE resets SET expires_at=? WHERE email=?', (int(time.time())-1,'al@cd.ef'))
        self.assertFalse(self.db.verify_otp('al@cd.ef', code, SECRET))

    def test_email_verification_before_login(self):
        ok, _ = self.db.signup('mariner', 'new@outlook.com', 'StrongPwd2!', QUESTION, 'Answer',
                               require_verification=True)
        self.assertTrue(ok)
        self.assertIsNone(self.db.authenticate('mariner','StrongPwd2!')[0])
        code, error = self.db.begin_email_verification('new@outlook.com', SECRET)
        self.assertIsNone(error)
        self.assertFalse(self.db.verify_email('new@outlook.com', '12345x', SECRET))
        self.assertTrue(self.db.verify_email('new@outlook.com', code, SECRET))
        self.assertFalse(self.db.verify_email('new@outlook.com', code, SECRET))
        self.assertIsNotNone(self.db.authenticate('mariner','StrongPwd2!')[0])
        self.assertEqual(self.db.user('mariner')['email_verified'], 1)

    def test_change_password(self):
        self.assertFalse(self.db.change_password('alex','bad','NewStrongPwd2!')[0])
        self.assertTrue(self.db.change_password('alex','StrongPwd1!','NewStrongPwd2!')[0])

    def test_tokens(self):
        token = issue_token('al@cd.ef', SECRET, version=3)
        self.assertEqual(decode_token(token,SECRET)['ver'],3)
        self.assertIsNone(decode_token(token, 'b'*48))
        expired = issue_token('al@cd.ef',SECRET,lifetime=-1)
        self.assertIsNone(decode_token(expired,SECRET))

    def test_delete(self):
        self.db.delete_user(self.db.user('alex')['id'])
        self.assertIsNone(self.db.user('alex'))


if __name__=='__main__':
    unittest.main()


In [ ]:
%%writefile tests/test_project.py
import unittest
from pathlib import Path

ROOT=Path(__file__).resolve().parents[1]

class ProjectTests(unittest.TestCase):
    def test_configuration_files_present(self):
        for fname in ['app.py','auth_core.py','requirements.txt','README.md','.env.example','Dockerfile']:
            self.assertTrue((ROOT/fname).exists(),fname)

    def test_missing_secrets_not_hardcoded(self):
        app=(ROOT/'app.py').read_text()
        self.assertIn('os.getenv("EMAIL_PASSWORD", "")',app)
        self.assertIn('os.getenv("ADMIN_PASSWORD", "")',app)
        self.assertIn('os.getenv("JWT_SECRET", "")',app)


if __name__=='__main__':
    unittest.main()


In [ ]:
!python -m unittest discover -s tests -v
!python -m py_compile auth_core.py app.py
from pathlib import Path
for filename in ('portwise_light_bg.png','portwise_dark_bg.png'):
    f=Path('static')/filename
    assert f.exists() and f.read_bytes().startswith(b'\x89PNG\r\n\x1a\n')
    print('Verified image:',filename)

## 6 · Check Gmail SMTP credentials (no test email sent)

This tests whether Gmail accepts the sender's App Password. It never prints the password. Actual inbox delivery must still be tested via the website with a registered email.

In [ ]:
import os, smtplib
if os.getenv('EMAIL_ADDRESS') and os.getenv('EMAIL_PASSWORD'):
    try:
        with smtplib.SMTP_SSL('smtp.gmail.com',465,timeout=15) as client:
            client.login(os.environ['EMAIL_ADDRESS'],os.environ['EMAIL_PASSWORD'])
        print('SMTP login accepted by Gmail. No email was sent.')
    except smtplib.SMTPAuthenticationError:
        print('SMTP rejected credentials. Recheck Gmail account and App Password.')
    except (smtplib.SMTPException,OSError) as err:
        print('SMTP connectivity failed:',type(err).__name__)
else:
    print('Set EMAIL_ADDRESS and EMAIL_PASSWORD under Colab Secrets to send OTPs.')

## 7 · Start the website and open the external ngrok link

This replaces the previous Streamlit server running on port 8501 **in this Colab runtime**, checks that the web page and two backgrounds respond, then starts a new tunnel. Keep Colab connected while the website is open.

If ngrok reports `ERR_NGROK_334`, stop the older endpoint in https://dashboard.ngrok.com/endpoints and retry this cell. Do not use tunnel pooling for independent app instances.

In [ ]:
import os, time, subprocess, signal
from getpass import getpass
from urllib.request import urlopen
from urllib.parse import urlencode
from IPython.display import display, HTML
from pyngrok import ngrok

try:
    old_pids=subprocess.check_output(['fuser','-n','tcp','8501'],stderr=subprocess.DEVNULL,text=True).split()
except (subprocess.CalledProcessError,FileNotFoundError):
    old_pids=[]
for pid in old_pids:
    if pid.isdigit() and int(pid)!=os.getpid():
        try: os.kill(int(pid),signal.SIGTERM)
        except ProcessLookupError: pass
time.sleep(2)

try:
    if streamlit_process.poll() is None:
        streamlit_process.terminate()
        streamlit_process.wait(timeout=10)
except (NameError,subprocess.TimeoutExpired):
    pass

with open('/tmp/portwise_final_streamlit.log','w') as log:
    streamlit_process=subprocess.Popen(
        ['streamlit','run','app.py','--server.port=8501',
         '--server.address=0.0.0.0','--server.headless=true',
         '--server.enableStaticServing=true','--browser.gatherUsageStats=false'],
        stdout=log,stderr=subprocess.STDOUT,env=os.environ.copy())

healthy=False
for _ in range(45):
    if streamlit_process.poll() is not None: break
    try:
        with urlopen('http://127.0.0.1:8501/_stcore/health',timeout=3) as r:
            healthy=r.status==200
        if healthy: break
    except Exception:
        pass
    time.sleep(1)
if not healthy:
    print(Path('/tmp/portwise_final_streamlit.log').read_text()[-3000:])
    raise RuntimeError('Streamlit server failed its health check')
for path in ('/','/app/static/portwise_light_bg.png','/app/static/portwise_dark_bg.png'):
    with urlopen('http://127.0.0.1:8501'+path,timeout=15) as r:
        prefix=r.read(100)
        assert r.status==200, (path,r.status)
        if path.endswith('.png'):
            assert prefix.startswith(b'\x89PNG\r\n\x1a\n'),path
    print('HTTP OK:',path)

token=os.environ.get('NGROK_AUTHTOKEN','').strip() or getpass('Paste your ngrok authtoken (hidden): ').strip()
if token:
    ngrok.set_auth_token(token)
    try:
        for t in ngrok.get_tunnels(): ngrok.disconnect(t.public_url)
    except Exception:
        pass
    try:
        tunnel=ngrok.connect(addr=8501,proto='http')
        base=tunnel.public_url.rstrip('/')
        print('PORTWISE is ready:',base)
        pages=[('Login','login'),('Signup','signup'),('Email verification','verify-email'),
               ('Forgot password','forgot'),('Admin login','admin'),
               ('Login demo','demo-login'),('OTP demo','demo-otp')]
        markup=['<h3>PORTWISE — Open an external website tab</h3>']
        for label,route in pages:
            url=base+'/?'+urlencode({'view':route})
            markup.append(f'<p><a href="{url}" target="_blank" rel="noopener noreferrer">{label} ↗</a></p>')
        display(HTML(''.join(markup)))
    except Exception as e:
        if 'ERR_NGROK_334' in str(e) or 'already online' in str(e):
            print('ngrok address is active elsewhere. Stop older ngrok sessions at https://dashboard.ngrok.com/endpoints and rerun this cell.')
        else:
            print('ngrok setup failed:',str(e)[:500])
else:
    print('Streamlit is running locally. Add NGROK_AUTHTOKEN for an external URL.')

## 8 · Final milestone verification

- Confirm the footer has `Build: PORTWISE-FINAL-01`.
- Confirm both maritime images and their gentle zoom remain visible in Light/Dark modes.
- Signup → email verification → login → dashboard → password change → logout.
- Forgot Password → Gmail OTP (countdown) and security question methods.
- Admin login → registered user directory and activity history.
- Capture your **own screenshots** including the OTP email, then redact sensitive data.
- Use **Runtime → Restart and Run all** to test setup from scratch. Clear cell outputs before submitting the `.ipynb` file.

**Hosting note:** Colab+ngrok is for the milestone demo. The repository also includes Docker, README, `.env.example`, tests and persistent-volume instructions for a dedicated host. Actual production launch still requires TLS, a real SMTP provider, durable data/backups and independent security review.